In [0]:
print("-----GIVEN DATAFRAME-----\n\n\n")

data = [
    ("a", [1, 1, 1, 3]),
    ("b", [1, 2, 3, 4]),
    ("c", [1, 1, 1, 1, 4]),
    ("d", [3]),
]

df = spark.createDataFrame(data, ["name", "rank"])

df.show()


print("-----REQUIRED DATAFRAME-----\n\n\n")


print("c\n\n")


print("-----DSL SOLUTION 1-----\n\n\n")


from pyspark.sql.functions import *
countdf = df.withColumn("count",expr("size(filter(rank,n->n==1))")).orderBy(col("count").desc())
countdf.show()

max_count = countdf.agg(max("count")).collect()[0][0]
print(max_count,"\n\n\n")

fildf = countdf.filter(col("count") == max_count).drop("count","rank").collect()[0][0]

print(fildf)


print("-----DSL SOLUTION 2-----\n\n\n") 

##This solution with explode is costlier at scale, and in the end returns only one matching row, not a great solution in case of ties



explodedf = df.withColumn("list",explode("rank")).drop("rank")
explodedf.show()


fildf = explodedf.filter(col("list")==1)
fildf.show()


countdf = fildf.groupBy("name").agg(count("*").alias("count")).orderBy(col("count").desc())
countdf.show()

finaldf = countdf.select(col("name")).first()[0]
print(finaldf)

